In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [2]:
import json
from uuid import uuid4
from pathlib import Path
from textwrap import dedent

import pandas as pd

from rich import print
from openai import OpenAI
from pydantic import BaseModel
from openai.lib._pydantic import to_strict_json_schema

In [4]:
client = OpenAI()

In [14]:
SYSTEM_PROMPTS = {
    "beir_corpus": dedent("""
                          Translate this Sundanese text including it's title and body into English.
                          Beware that it might contain a specific Sundanese context or nuances that must be correctly interpreted and not translated literally.
                          """),
    "beir_query": dedent("""
                         Translate this Sundanese with possible Indonesian text into English.
                         Beware that it might contain a specific Sundanese context or nuances that must be correctly interpreted and not translated literally.
                         """),
    "triplet": dedent("""
                      Translate this Sundanese passages into English.
                      You will be provided with a query along with the relevant and irrelevant answers.
                      Beware that it might contain a specific Sundanese context or nuances that must be correctly interpreted and not translated literally.
                      """),
}

## BEIR

### Corpus

In [6]:
df_corpus = pd.read_json("../data/beir/corpus.jsonl", lines=True)
df_corpus.head()

,_id,title,text
0,0f438470-de7f-47cb-8ba2-16e8b1ff5750,JEMBAR SABAR,bismillah yuga lampah balukar janglar meunang ...
1,c51ddd60-adc7-4b95-b09d-3c4865ba2aaf,KANYERI,eweuh deui seri nu lewih nyeri tibatan di héna...
2,cb4eb9c6-bf7f-4edc-b997-73a02780e60d,KARUNGING LAIN WAYAH,wanci gayuh ka peuting poék tungkeb haté lain ...
3,af9fc2c2-0230-4784-8104-cf9271ccfccc,KUDU DAÉK GAWÉ,ieu awak asa lalungsé di rasa asa carapé meure...
4,2ae6c860-37c6-47f0-aefc-8b74f96b623b,PUISI PANGGGEUING DIRI,naha anjeun téh poho yén mot téh dodoho datang...


In [19]:
def corpus_item_text(value):
    return "<title>" + value.title + "</title>\n<body>" + value.text + "</body>"

In [7]:
class BEIRCorpusItem(BaseModel):
    title_english: str
    body_english: str

In [ ]:
completion_beir_corpus = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=BEIRCorpusItem,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["beir_corpus"],
        },
        {
            "role": "user",
            "content": corpus_item_text(df_corpus.iloc[0]),
        },
    ],
)

print(completion_beir_corpus)

ParsedChatCompletion[BEIRCorpusItem](
    id='chatcmpl-BSkICIleovtteNlCDnUmNbf8HajwW',
    choices=[
        ParsedChoice[BEIRCorpusItem](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[BEIRCorpusItem](
                content='{"title_english":"WIDE PATIENCE","body_english":"In the name of God, start to take a step 
towards achieving desires, avoiding distractions. Be patient, do not be indifferent; cultivate a thoughtful spirit.
When unwell, do not fret; when healthy, do not take it for granted. Do not whine and become small, do not be 
arrogant in a big place. Everyone is a relative seeking halal sustenance. Do not be confused by the worldly 
pleasures; always align yourself with the Almighty. Understand what is true and examine your heart."}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=BEIRCorpusItem(
                    title_english='WIDE PATIENCE',
                    body_english='In the name of God, start to take a step towards achieving desires, avoiding 
distractions. Be patient, do not be indifferent; cultivate a thoughtful spirit. When unwell, do not fret; when 
healthy, do not take it for granted. Do not whine and become small, do not be arrogant in a big place. Everyone is 
a relative seeking halal sustenance. Do not be confused by the worldly pleasures; always align yourself with the 
Almighty. Understand what is true and examine your heart.'
                ),
                annotations=[]
            )
        )
    ],
    created=1746190832,
    model='gpt-4o-mini-2024-07-18',
    object='chat.completion',
    service_tier='default',
    system_fingerprint='fp_dbaca60df0',
    usage=CompletionUsage(
        completion_tokens=117,
        prompt_tokens=230,
        total_tokens=347,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)

### Queries

In [9]:
df_queries = pd.read_json("../data/beir/queries.jsonl", lines=True)
df_queries.head()

,_id,text
0,f4ee6408-7047-498e-a319-9189cf81d378,apa maksud dari bismillah yuga lampah
1,c827cfdb-2c57-486f-abc7-80355b884699,kenapa penting sabar dalam hidup
2,0dce18d7-b0b8-4202-a1d5-683489ac423d,apa yang dimaksud dengan halangan dalam mencap...
3,5c961ec5-158a-4073-b33e-3a0be48de7a0,bagaimana cara menjadi orang yang baik menurut...
4,3a019977-ed9b-4d71-8dbd-a65c1c95cc65,apa arti dari pariksa ati


In [15]:
class BEIRQueryItem(BaseModel):
    query_english: str

In [17]:
completion_beir_query = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=BEIRQueryItem,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["beir_query"],
        },
        {
            "role": "user",
            "content": df_queries.iloc[0, 1],
        },
    ],
)

print(completion_beir_query)

ParsedChatCompletion[BEIRQueryItem](
    id='chatcmpl-BSkSNV5nGwRkRJEMBYprNyxdnbS6N',
    choices=[
        ParsedChoice[BEIRQueryItem](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[BEIRQueryItem](
                content='{"query_english":"What does \'bismillah yuga lampah\' mean?"}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=BEIRQueryItem(query_english="What does 'bismillah yuga lampah' mean?"),
                annotations=[]
            )
        )
    ],
    created=1746191463,
    model='gpt-4o-mini-2024-07-18',
    object='chat.completion',
    service_tier='default',
    system_fingerprint='fp_dbaca60df0',
    usage=CompletionUsage(
        completion_tokens=20,
        prompt_tokens=101,
        total_tokens=121,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)

## Triplet

In [18]:
df_triplet = pd.read_json("../data/triplet/triplet.jsonl", lines=True)
df_triplet.head()

,query,positive,negative
0,Kumaha cara ngahontal kahayang dina kahirupan?,"Kahiji, urang kedah sabar sareng henteu janten...",Abdi ngadangu seueur warta ngeunaan jalma anu ...
1,Naon anu kedah dilakukeun lamun gering?,"Lamun gering, ulah rungsing sabab pikiran posi...",Masyarakat ayeuna seueur nganggur di kota nu g...
2,Kumaha cara nyieun amal?,"Ngawitan amal ti hal-hal leutik, sapertos ngab...",Jalma sering nyarita ngeunaan kaékonomian anu ...
3,Kumaha sangkan ngeterkeun diri ka Gusti?,"Mertahankeun ati, pariksa tindakan sorangan, s...","Saurang guru ngajarkeun pentingna ilmu, tapi k..."
4,Naon hartina jadi jalma leutik?,Jadi jalma leutik hartina ulah sombong sareng ...,"Dina pagelaran, anu katinggali gaduh prestasi ..."


In [21]:
def triplet_item_text(value):
    return "<query>" + value.query + "</query>\n<positive>" + value.positive + "</positive>\n<negative>" + value.negative + "</negative>"

In [23]:
class TripletItem(BaseModel):
    query_english: str
    positive_english: str
    negative_english: str

In [24]:
completion_triplet = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=TripletItem,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["beir_query"],
        },
        {
            "role": "user",
            "content": triplet_item_text(df_triplet.iloc[0]),
        },
    ],
)

print(completion_triplet)

ParsedChatCompletion[TripletItem](
    id='chatcmpl-BSkXIbWYaRijcSfCK4rOnBC4ZhUNq',
    choices=[
        ParsedChoice[TripletItem](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[TripletItem](
                content='{"query_english":"How to achieve desires in life?","positive_english":"First, we must be 
patient and not lose enthusiasm in our efforts, because the consequences of perseverance required to achieve 
desires need diligent effort.","negative_english":"I\'ve heard a lot of news about successful people, but have 
never heard how they overcame obstacles."}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=TripletItem(
                    query_english='How to achieve desires in life?',
                    positive_english='First, we must be patient and not lose enthusiasm in our efforts, because the
consequences of perseverance required to achieve desires need diligent effort.',
                    negative_english="I've heard a lot of news about successful people, but have never heard how 
they overcame obstacles."
                ),
                annotations=[]
            )
        )
    ],
    created=1746191768,
    model='gpt-4o-mini-2024-07-18',
    object='chat.completion',
    service_tier='default',
    system_fingerprint='fp_dbaca60df0',
    usage=CompletionUsage(
        completion_tokens=69,
        prompt_tokens=209,
        total_tokens=278,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)

## TODO: Generate OpenAI Batch Request

### Batch Request Generator

In [ ]:
def generate_batch(df: pd.DataFrame, system_prompt: str, base_model: BaseModel):
    for row in df.itertuples():
        custom_id = str(uuid4())
        job_data = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-4o-mini",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": row.content},
                ],
                "response_format": {
                    "type": "json_schema",
                    "json_schema": {
                        "name": base_model.__name__,
                        "strict": True,
                        "schema": to_strict_json_schema(base_model),
                    },
                },
            },
        }
        
        yield (custom_id, row.id, job_data)

In [ ]:
next(generate_batch(df, SYSTEM_PROMPTS["BEIR"], BEIRQuery))

In [ ]:
def persist_batch(kind: str, schema: BaseModel):
    batch_req_path = Path(f"../data/{kind}/{kind}_batch.jsonl")
    batch_req_path.parent.mkdir(parents=True, exist_ok=True)

    batch_map_path = Path(f"../data/{kind}/{kind}_map.jsonl")
    batch_map_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(batch_req_path, "w") as fm, open(batch_map_path, "w") as mm:
        batch_iter = generate_batch(df, SYSTEM_PROMPTS[kind.upper()], schema)
        for custom_id, doc_id, req in batch_iter:
            json.dump(req, fm)
            fm.write("\n")

            json.dump({"custom_id": custom_id, "doc_id": doc_id}, mm)
            mm.write("\n")
    
    return batch_req_path, batch_map_path

In [ ]:
beir_req_path, beir_map_path = persist_batch("beir", BEIRQuery)
triplet_req_path, triplet_map_path = persist_batch("triplet", TripetData)

### Submit Batch Requests

In [ ]:
def submit_batch(path):
    batch_file = client.files.create(file=open(path, "rb"), purpose="batch")

    return client.batches.create(
        input_file_id=batch_file.id, 
        endpoint="/v1/chat/completions", 
        completion_window="24h"
    )

In [ ]:
beir_batch = submit_batch(beir_req_path.resolve())
print(beir_batch)

In [ ]:
triplet_batch = submit_batch(triplet_req_path.resolve())
print(triplet_batch)